# Feature Engineering

Generates two feature artefacts consumed by the model building step.

| File | Key | Content |
|---|---|---|
| `features/ds_features.parquet` | `unique_id x ds` | Cyclic calendar encodings, product attributes, entity IDs |
| `features/ds_holidays.parquet` | `ds x region_id` | National and regional holiday flags, type, proximity |

**Static features** (`ds_features`): constant per `unique_id`, valid for any forecast week.
**Holiday features** (`ds_holidays`): dynamic per `ds`, vary by macro-region (5 regions -> Brazilian states).

Distribution statistics (cv, iqr, q50) are computed in model building after the train/test split
to avoid data leakage.

In [ ]:
import os
import unicodedata
import pandas as pd
import numpy as np
import warnings
from holidays import country_holidays
warnings.filterwarnings('ignore')

In [ ]:
# -- Paths -----------------------------------------------------------------
PATH_REFINED    = '../../data/gold/forecasting/datasets/refined'
PATH_BASE       = '../../data/gold/forecasting/datasets/base/ds_sales_timeseries.parquet'
PATH_DIM_REGION = '../../data/gold/data_warehouse/dw_dim_region.parquet'

PATH_FEATURES   = '../../data/gold/forecasting/features/ds_features.parquet'
PATH_HOLIDAYS   = '../../data/gold/forecasting/features/ds_holidays.parquet'
os.makedirs(os.path.dirname(PATH_FEATURES), exist_ok=True)

DEMAND_TYPES = ['smooth', 'erratic', 'intermittent', 'lumpy']

## 1. Load Refined Series

In [ ]:
df_valid = pd.concat([
    pd.read_parquet(f'{PATH_REFINED}/ds_sales_timeseries_{dt}.parquet')
    for dt in DEMAND_TYPES
], ignore_index=True)

print(f'Valid series : {df_valid["unique_id"].nunique():,}')
print(f'Total rows   : {len(df_valid):,}')

## 2. Load Static Attributes

In [ ]:
# Load only attribute columns - stats excluded to prevent leakage
BASE_COLS = [
    'time_series_id', 'week_date',
    'week', 'month', 'quarter', 'year',
    'product_attribute_1', 'product_attribute_2', 'product_attribute_3',
    'supplier_id', 'region_id',
]
df_base = pd.read_parquet(PATH_BASE, columns=BASE_COLS)
print(f'Base attributes loaded: {df_base.shape}')

## 3. Build Features

In [ ]:
def build_calendar_features(df: 'pd.DataFrame') -> 'pd.DataFrame':
    # Cyclic encoding: preserves circular proximity (month 12 near month 1)
    df['month_sin']   = np.sin(2 * np.pi * df['month']   / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month']   / 12)
    df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
    df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)
    df['week_sin']    = np.sin(2 * np.pi * df['week']    / 52)
    df['week_cos']    = np.cos(2 * np.pi * df['week']    / 52)
    return df.drop(columns=['week', 'month', 'quarter'])  # year kept as numeric


def build_product_features(df: 'pd.DataFrame') -> 'pd.DataFrame':
    for col in ['product_attribute_1', 'product_attribute_2', 'product_attribute_3']:
        df[col] = df[col].astype('category').cat.codes
    return df


def build_features(df_valid: 'pd.DataFrame', df_base: 'pd.DataFrame') -> 'pd.DataFrame':
    df = df_valid[['unique_id', 'ds']].copy()
    df = df.merge(
        df_base.rename(columns={'time_series_id': 'unique_id', 'week_date': 'ds'}),
        on=['unique_id', 'ds'], how='left',
    )
    df = build_calendar_features(df)
    df = build_product_features(df)
    return df


df_features = build_features(df_valid, df_base)

print(f'Shape   : {df_features.shape}')
print(f'Columns : {df_features.columns.tolist()}')
if df_features.isna().any().any():
    print('NaN (%) :')
    print(df_features.isna().mean().sort_values(ascending=False).head(5))

## 4. Save -- Static Features

In [ ]:
df_features.to_parquet(PATH_FEATURES, index=False)

cal_cols  = [c for c in df_features.columns if any(s in c for s in ['sin', 'cos', 'year'])]
prod_cols = [c for c in df_features.columns if 'attribute' in c]
id_cols   = ['supplier_id', 'region_id']

print(f'Saved   : {PATH_FEATURES}')
print(f'Shape   : {df_features.shape}')
print(f'Calendar: {cal_cols}')
print(f'Product : {prod_cols}')
print(f'IDs     : {id_cols}')

## 5. Holiday Features

Holiday features keyed by `(ds, region_id)`.

**Macro-region to states:**

| region_id | Region | States |
|---|---|---|
| 1 | Centro-Oeste | DF, GO, MS, MT |
| 2 | Nordeste | AL, BA, CE, MA, PB, PE, PI, RN, SE |
| 3 | Norte | AC, AM, AP, PA, RO, RR, TO |
| 4 | Sudeste | ES, MG, RJ, SP |
| 5 | Sul | PR, RS, SC |

**Columns generated:**
- `is_national_holiday` - national holiday falls within the week
- `is_regional_holiday` - state-specific holiday exclusive to this region
- `is_holiday` - any holiday (national or regional)
- `n_holidays` - total holidays in the week
- `holiday_type` - dominant type: `none / regional / other / christmas_newyear / easter / carnival`
- `holiday_type_enc` - ordinal encoding by expected demand impact (0=none to 5=carnival)
- `weeks_to_next_holiday` - weeks to the next holiday, capped at 4
- `weeks_since_last_holiday` - weeks since the last holiday, capped at 4

In [ ]:
REGION_TO_STATES = {
    1: ['DF', 'GO', 'MS', 'MT'],
    2: ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
    3: ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO'],
    4: ['ES', 'MG', 'RJ', 'SP'],
    5: ['PR', 'RS', 'SC'],
}

# Priority ordered by expected pharmaceutical demand impact
HOLIDAY_TYPE_KEYWORDS = [
    ('carnival',          ['carnaval', 'quaresma']),
    ('easter',            ['sexta-feira santa', 'corpus christi']),
    ('christmas_newyear', ['natal', 'ano-novo', 'confraterniza']),
    ('other',             []),
]
HOLIDAY_TYPE_ENC = {
    'none': 0, 'regional': 1, 'other': 2,
    'christmas_newyear': 3, 'easter': 4, 'carnival': 5,
}


def _strip_accents(text: str) -> str:
    nfkd = unicodedata.normalize('NFD', text.lower())
    return ''.join(c for c in nfkd if unicodedata.category(c) != 'Mn')


def classify_holiday_type(names: list) -> str:
    combined = _strip_accents(' '.join(names))
    for htype, keywords in HOLIDAY_TYPE_KEYWORDS:
        if keywords and any(k in combined for k in keywords):
            return htype
    return 'other' if names else 'none'


def build_holiday_features(all_weeks: 'pd.Series') -> 'pd.DataFrame':
    years    = all_weeks.dt.year.unique().tolist()
    all_rows = []

    for region_id, states in REGION_TO_STATES.items():
        national_hols = country_holidays('BR', years=years)
        regional_hols = set()
        for state in states:
            try:
                state_hols = country_holidays('BR', subdiv=state, years=years)
                for d, name in state_hols.items():
                    if d not in national_hols:
                        regional_hols.add((d, name))
            except Exception:
                pass

        for week_start in all_weeks:
            week_days = pd.date_range(week_start, periods=7, freq='D')
            national_in_week = [(d.date(), n) for d, n in national_hols.items()
                                if d in week_days.date]
            regional_in_week = [(d, n) for d, n in regional_hols
                                if d in week_days.date]

            is_national = len(national_in_week) > 0
            is_regional = len(regional_in_week) > 0
            is_holiday  = is_national or is_regional
            n_holidays  = len(national_in_week) + len(regional_in_week)

            if is_national:
                names = [n for _, n in national_in_week]
                htype = classify_holiday_type(names)
            elif is_regional:
                htype = 'regional'
            else:
                htype = 'none'

            all_rows.append({
                'ds':                  week_start,
                'region_id':           region_id,
                'is_national_holiday': is_national,
                'is_regional_holiday': is_regional,
                'is_holiday':          is_holiday,
                'n_holidays':          n_holidays,
                'holiday_type':        htype,
                'holiday_type_enc':    HOLIDAY_TYPE_ENC.get(htype, 0),
            })

    df_h = pd.DataFrame(all_rows).sort_values(['ds', 'region_id']).reset_index(drop=True)

    # Proximity features
    for region_id in df_h['region_id'].unique():
        mask   = df_h['region_id'] == region_id
        df_r   = df_h[mask].copy().reset_index()
        hol_idx = df_r.index[df_r['is_holiday']].tolist()
        to_next    = []
        since_last = []
        for i in range(len(df_r)):
            future = [j - i for j in hol_idx if j >= i]
            past   = [i - j for j in hol_idx if j <= i]
            to_next.append(min(future) if future else 4)
            since_last.append(min(past) if past else 4)
        df_h.loc[mask, 'weeks_to_next_holiday']    = [min(v, 4) for v in to_next]
        df_h.loc[mask, 'weeks_since_last_holiday'] = [min(v, 4) for v in since_last]

    return df_h


all_weeks   = df_valid['ds'].drop_duplicates().sort_values().reset_index(drop=True)
df_holidays = build_holiday_features(all_weeks)

print(f'Shape   : {df_holidays.shape}')
print(f'Regions : {sorted(df_holidays["region_id"].unique())}')
print(f'Weeks   : {df_holidays["ds"].nunique()}')
print()
print('Holidays by type (region 4 - Sudeste):')
print(df_holidays[df_holidays['region_id'] == 4]
      .groupby('holiday_type')['is_holiday'].sum()
      .sort_values(ascending=False).to_string())

## 6. Save -- Holiday Features

In [ ]:
df_holidays.to_parquet(PATH_HOLIDAYS, index=False)

print(f'Saved : {PATH_HOLIDAYS}')
print(f'Shape : {df_holidays.shape}')
print()
print('Column groups:')
print('  Key         : ["ds", "region_id"]')
print('  Flags       : ["is_national_holiday", "is_regional_holiday", "is_holiday", "n_holidays"]')
print('  Type        : ["holiday_type", "holiday_type_enc"]')
print('  Proximity   : ["weeks_to_next_holiday", "weeks_since_last_holiday"]')